In [1]:
from datasets import Dataset, DatasetDict
from setfit import SetFitModel, Trainer, TrainingArguments, sample_dataset
import pickle 
import pandas as pd
from tqdm import tqdm
import numpy as np
from rich import print
import mlflow 

In [4]:

model = SetFitModel.from_pretrained(r"C:\Users\jvhua\OneDrive\Desktop\ISYE-CSE-MGT-6748-Group-1\setfit_model_v5")


In [5]:
with open(r'C:\Users\jvhua\OneDrive\Desktop\ISYE-CSE-MGT-6748-Group-1\02_preprocess\unclassified_industry_batch12_v2.pickle', 'rb') as file:
    data = pickle.load(file)

In [6]:
data1 = [(k, v) for k, vals in data.items() for v in vals]
data1 = pd.DataFrame(data1, columns=['uuid', 'text'])

In [7]:
labels = [model.predict([sent]) for sent in tqdm(data1['text'])]

100%|██████████| 13750/13750 [06:04<00:00, 37.71it/s]


In [8]:
data1['pred_label'] = labels
data1['preds'] = data1['pred_label'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else x)

In [9]:
filtered_data = data1[data1['preds'] != 0]
non_signal_data = data1[data1['preds'] == 0]
data1_filtered = filtered_data.groupby('uuid')['text'].apply(' '.join).reset_index()
data1_filtered_no_signal = non_signal_data.groupby('uuid')['text'].apply(' '.join).reset_index()

In [10]:
uuid_list = data1_filtered['uuid'].tolist() + data1_filtered_no_signal['uuid'].tolist()

In [12]:
original_data = pd.read_csv(r"C:\Users\jvhua\OneDrive\Desktop\ISYE-CSE-MGT-6748-Group-1\data\unclassified_postings.csv", index_col =0)

In [13]:
filtered_original_data = original_data[original_data['id'].isin(uuid_list)]

In [14]:
def count_words(text):
    words = text.split()
    return len(words)
filtered_original_data['word_count'] = filtered_original_data['body'].apply(count_words)

C:\Users\jvhua\AppData\Local\Temp\ipykernel_11712\2156550060.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_original_data['word_count'] = filtered_original_data['body'].apply(count_words)


In [15]:
data1_filtered['word_count_signal'] = data1_filtered['text'].apply(count_words)
data1_filtered_no_signal['word_count_no_signal'] = data1_filtered_no_signal['text'].apply(count_words)

In [20]:
new_data = data1_filtered.merge(filtered_original_data, left_on = 'uuid', right_on = 'id', how = 'left')
new_data1 = new_data.merge(data1_filtered_no_signal, left_on = 'uuid', right_on = 'uuid', how = 'left')

In [27]:
pd.set_option('display.max_colwidth', None)

In [29]:
new_data1[['body', 'text_y', 'text_x']].sample(5)

,body,text_y,text_x
59,"CURRENT EMPLOYEES - Please apply using the career worklet in Workday. This career site is for external applicants only. Employee Type: Seasonal (Seasonal) Do you want the opportunity to work with some of the largest equipment in America? The Coteau Properties Company is looking for talented maintenance welders to join our award-winning team at the Freedom Mine. Our seasonal dragline maintenance welders are employed eight to nine months during the year with the potential to become a full-time employee. The Coteau Properties Company, a subsidiary of North American Coal Corporation, operates the Freedom Mine, the largest lignite surface coal mine in the United States with large electric draglines, loading shovels, and coal preparation facilities located north of Beulah, North Dakota. We empower each and every employee with safety at the forefront of everything we do. Job Description: Qualified applicants must be must be able to perform all aspects of welding and mechanical repairs to draglines, shovels and assorted mining equipment. Must have knowledge of all phases of welding, cutting, air arcing, grinding, and metal properties. Applicants must have good communication, interpersonal skills, accept responsibility and be team oriented. Applicants must be willing to work a 12-hour shift schedule. Welding and maintenance experience and an associates degree in mechanical maintenance or welding is preferred. We offer a competitive salary starting at $46.60/hour and a medical benefits package for seasonal employees, including 401K options. Eligible benefits are effective on the first day of employment. If you are ready for a career change and are excited to be part of our team, please complete the application process and include a resume online at www.nacoal.com/careers. The position closes January 24, 2024. The Coteau Properties Company Human Resources Department 204 County Road 15 Beulah, ND 58523 701-873-2281 EEO Statement: EEO/M/F/disability/veteran You must meet at least the specific minimum requirements listed above to be considered a qualified applicant. We're fortunate to have some of the most talented and experienced professionals in the coal mining industry on our team. We trust and empower our people to always do the right thing, look out for one another, and draw on their collective expertise to provide efficient and cost-effective solutions for every customer.","CURRENT EMPLOYEES - Please apply using the career worklet in Workday. This career site is for external applicants only. We empower each and every employee with safety at the forefront of everything we do. Applicants must have good communication, interpersonal skills, accept responsibility and be team oriented. Eligible benefits are effective on the first day of employment. If you are ready for a career change and are excited to be part of our team, please complete the application process and include a resume online at www.nacoal.com/careers. The position closes January 24, 2024. We're fortunate to have some of the most talented and experienced professionals in the coal mining industry on our team. We trust and empower our people to always do the right thing, look out for one another, and draw on their collective expertise to provide efficient and cost-effective solutions for every customer.","Employee Type: Seasonal (Seasonal) Do you want the opportunity to work with some of the largest equipment in America? The Coteau Properties Company is looking for talented maintenance welders to join our award-winning team at the Freedom Mine. Our seasonal dragline maintenance welders are employed eight to nine months during the year with the potential to become a full-time employee. The Coteau Properties Company, a subsidiary of North American Coal Corporation, operates the Freedom Mine, the largest lignite surface coal mine in the United States with large electric draglines, loading shovels, and coal preparation facilities located north of Beulah, Nor

In [ ]:
#should go by order number 
#put into function 